# Collecting the records of InCites Journal Citation Reports (Web of Science)

In [ ]:
# Importing the required libraries.
import re, traceback, csv, pandas as pd, time, os
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, TimeoutException, StaleElementReferenceException
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service

## 1. Defining the class of Crawler

In [ ]:
class InCitesSpider:
    def __init__(self, url, login, password, driver_path):
        self.__url_base = url
        self.__username = login
        self.__password = password
        self.__driver_path = driver_path
        self.__data = None
        self.__driver = None
        os.environ["USER_AUTHENTICATED"] = "False"

    @property
    def get_data(self):
        return self.__data

    def __init_webdriver(self, is_firefox=True):
        # Choosing the webdriver.
        if not is_firefox and self.__driver is None:
            # Running the PhantomJS webdriver.
            self.__driver = webdriver.PhantomJS()
            self.__driver.set_window_size(1120, 550)
        elif self.__driver is None:
            # Defining the option to the Firefox webdriver.
            options = Options()
            options.set_preference("headless", False)

            # Running the Firefox webdriver.
            service = Service(executable_path = self.__driver_path)
            self.__driver = webdriver.Firefox(service=service, options=options)

            if is_firefox & options.preferences["headless"]:
                self.__driver.set_window_size(1120, 550)
                self.__driver.maximize_window()

        # Getting the web page.
        self.__driver.get(self.__url_base)

    def __authenticate(self):
        # Waiting for 10 seconds.
        WebDriverWait(self.__driver, 10).until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "input#mat-input-1")))

        # Authenticating with user's account data.
        username_field = self.__driver.find_element(
            By.CSS_SELECTOR, "input#mat-input-1")
        password_field = self.__driver.find_element(
            By.CSS_SELECTOR, "input#mat-input-0")
        username_field.send_keys(self.__username)
        password_field.send_keys(self.__password)
        password_field.send_keys(Keys.RETURN)

        # Enabling the cookies.
        time.sleep(10)
        button = self.__driver.find_element(By.ID, "onetrust-accept-btn-handler")
        WebDriverWait(self.__driver, 120).until(EC.element_to_be_clickable(button))
        button.click()

        # Redirecting the list of journals.
        time.sleep(10)
        button = self.__driver.find_element(By.CSS_SELECTOR, "a[href='/jcr/browse-journals']")
        self.__driver.execute_script("arguments[0].scrollIntoView();", button)
        WebDriverWait(self.__driver, 120).until(EC.element_to_be_clickable(button))
        button.click()

        os.environ["USER_AUTHENTICATED"] = "True"

    def __parse_items(self):
        num_item_per_page = None
        flag = True
        self.__data = list()

        # Getting the number of journals.
        time.sleep(10)
        element = self.__driver.find_element(By.CSS_SELECTOR, "p.journal-count-header")
        WebDriverWait(self.__driver, 30).until(EC.visibility_of(element))
        num_journals = element.text
        num_journals = int(num_journals.replace(",", "").split(" ")[0])

        # Setting the number of items per page.
        css = "mat-select[aria-label='Items per page:']"
        element = self.__driver.find_element(By.CSS_SELECTOR, css)
        WebDriverWait(self.__driver, 30).until(EC.element_to_be_clickable(element)).click()
        element = self.__driver.find_element(
            By.CSS_SELECTOR, f"div[aria-label='Items per page:'] > mat-option:nth-child(5)")
        element.click()
        css = f"{css} > div > div:first-child > span > span"
        element = self.__driver.find_element(By.CSS_SELECTOR, css)
        num_item_per_page = int(element.text)
        print("Number of Items per Page:", num_item_per_page)

        while flag:
            # Showing the progress.
            print(f"Collected: {len(self.__data)} of {num_journals} ({((len(self.__data) / num_journals) * 100):.2f}%)")
            time.sleep(10)
            try:
                # Waiting to load the records.
                WebDriverWait(self.__driver, 120).until(EC.invisibility_of_element(
                    (By.CSS_SELECTOR, "div.backdrop")))

                # Waiting to load the table of records.
                WebDriverWait(self.__driver, 120).until(EC.visibility_of_element_located(
                    (By.CSS_SELECTOR, "section.table-section > mat-table[class*='mat-table']")))

                # Defining the scraper.
                html_soup = BeautifulSoup(self.__driver.page_source, "html.parser")

                # Getting the rows.
                rows = html_soup.find_all("mat-row")
                for idx, row in enumerate(rows):
                    try:
                        record = dict()
                        # Getting the columns/cells of data.
                        cells = row.select("mat-cell > span")

                        # Journal name.
                        record["journal_name"] = re.sub(r"\s+", " ", cells[0].string).strip()

                        # ISSN.
                        record["issn"] = re.sub(r"\s+", " ", cells[1].string).strip()

                        # eISSN.
                        record["e_issn"] = re.sub(r"\s+", " ", cells[2].string).strip()

                        # Category and Edition.
                        css = "span.multiple > mat-expansion-panel > div:nth-child(2) > div > div > span"
                        if cells[3].select(css):
                            record["category"] = tuple([re.sub(r"\s+", " ", item.string).strip()
                                                        for item in cells[3].select(css)])
                            css = "span.table-cell-edition > mat-expansion-panel > div:nth-child(2) > div > span"
                            record["edition"] = tuple([re.sub(r"\s+", " ", item.string).strip()
                                                       for item in cells[4].select(css)])
                        else:
                            record["category"] = re.sub(r"\s+", " ", cells[3].string).strip()
                            record["edition"] = tuple([re.sub(r"\s+", " ", item).strip()
                                                       for item in cells[4].string.split(",")])

                        # Total citations.
                        record["total_citations"] = re.sub(r"\s+", " ", cells[5].string).strip()

                        # 2025 JIF.
                        record["impact_factor_2025"] = re.sub(r"\s+", " ", cells[6].string).strip()

                        # JIF Quartile.
                        css = "span.multiple > mat-expansion-panel > div:nth-child(2) > div > span"
                        record["jif_quartile"] = tuple([re.sub(r"\s+", " ", item.string).strip()
                                                        for item in cells[7].select(css)]) \
                            if cells[7].select(css) else re.sub(r"\s+", " ", cells[7].string).strip()

                        # 2025 JCI.
                        record["jci_2025"] = re.sub(r"\s+", " ", cells[8].string).strip()

                        # % of Citable OA.
                        record["percent_citable_oa"] = re.sub(r"\s+", " ", cells[9].string).strip()

                        self.__data.append(record)
                    except Exception as e:
                        print(idx)
                        raise e

                # Clicking the button.
                button = self.__driver.find_element(By.CSS_SELECTOR, "button.mat-paginator-navigation-next")
                WebDriverWait(self.__driver, 120).until(EC.element_to_be_clickable(button))
                self.__driver.execute_script("arguments[0].scrollIntoView();", button)
                flag = False if button.get_attribute("disabled") else True
                if flag:
                    button.click()
            except (NoSuchElementException, TimeoutException, StaleElementReferenceException) as e:
                print(f"[ERROR-DEBUG] {e}: {self.__url_base}")
                print("".join(traceback.format_tb(e.__traceback__)))
                flag = False

    def collect(self):
        # Getting webdriver.
        self.__init_webdriver()

        # Authenticating the valid user and enabling the cookies.
        self.__authenticate()
        is_authenticated = bool(os.environ["USER_AUTHENTICATED"])

        # Crawling the data.
        if is_authenticated:
            self.__parse_items()

        # Closing the webdriver.
        self.__driver.quit()

## 2. Getting the data from its URL

In [ ]:
# Defining the credentials.
username = ">>> PUT YOUR E-MAIL <<<"
password = ">>> PUT YOUR PASSWORD <<<"

# Determining the URL of target page.
url = "https://jcr.clarivate.com/"

# Defining the driver path.
driver_path = "/Users/breno/geckodriver/geckodriver"

# Creating the spider.
spider = InCitesSpider(url, username, password, driver_path)

# Collecting the data.
spider.collect()

In [ ]:
# Getting the collected data.
data = spider.get_data

In [ ]:
# Printing the number of records collected.
print("Number of records collected: {}.".format(len(data)))

## 3. Saving the data collected

In [ ]:
# Creating the Pandas' DataFrame object.
df_data = pd.DataFrame(data)

In [ ]:
# Preprocessing the data.
df_data.replace({"n/a": None, "N/A": None}, inplace=True)
df_data[["category", "edition", "jif_quartile"]] = df_data[["category", "edition", "jif_quartile"]].apply(
    lambda x: x.apply(lambda y: tuple(y) if type(y) == list else y), axis=1)
df_data.drop_duplicates(keep="first", inplace=True)

In [ ]:
# Checking the information about the dataset.
df_data.info()

In [ ]:
# Exporting the data to CSV file.
df_data.to_csv("jcr_2025_wos.csv", index=False, quoting=csv.QUOTE_ALL)